# L200 · Lakebase setup for agent memory + the Action Plane

This notebook prepares Lakebase (managed Postgres) for two things the L200 agent uses:

1. **Short term memory** for the OpenAI Agents SDK agent, in the `agent_openai_memory` schema. The agent server creates these tables itself on first start, so you do not run that here.
2. **The Action Plane** tables (`actions`, `action_events`, `action_policies`) in the `akzo_actions` schema, created by `scripts/actions_schema_setup.py`.

Nothing about a workspace is hardcoded. You set your Lakebase instance name once, as a widget, and everything else follows. On a Vocareum lab this is your assigned instance.

> Note: all data is synthetic. This is a workshop environment.

In [ ]:
%pip install "databricks-openai[memory]>=0.13.0" "databricks-sdk>=0.70.0" "databricks-ai-bridge"
dbutils.library.restartPython()

## 0. Parameters

The one value you must set is your Lakebase instance name. The schema names default to the workshop conventions and rarely need changing.

In [ ]:
import re

dbutils.widgets.text("lakebase_instance", "", "Lakebase instance name")
dbutils.widgets.text("memory_schema", "agent_openai_memory", "Agent memory schema")
dbutils.widgets.text("actions_schema", "akzo_actions", "Action Plane schema")

lakebase_instance = dbutils.widgets.get("lakebase_instance").strip()
memory_schema = dbutils.widgets.get("memory_schema").strip()
actions_schema = dbutils.widgets.get("actions_schema").strip()

if not lakebase_instance:
    raise ValueError(
        "Set the 'lakebase_instance' widget to your Lakebase instance name "
        "(on a Vocareum lab, your assigned instance)."
    )
for name in (memory_schema, actions_schema):
    if not re.fullmatch(r"[A-Za-z0-9_]+", name):
        raise ValueError(f"Unsafe schema name: {name!r}.")

print(f"Lakebase instance: {lakebase_instance}")
print(f"Memory schema:     {memory_schema}")
print(f"Actions schema:    {actions_schema}")

## 1. Confirm the instance is reachable

We look the instance up through the SDK to confirm the name resolves and you can reach it. This is also where a clear preflight error surfaces if the instance is missing or you lack permission, rather than failing later mid run.

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
inst = w.database.get_database_instance(name=lakebase_instance)
print(f"Found instance '{inst.name}'")
print(f"  read/write DNS: {inst.read_write_dns}")
print(f"  state:          {getattr(inst, 'state', 'n/a')}")

## 2. Agent memory schema

The OpenAI Agents SDK agent server creates its own memory tables in `agent_openai_memory` on first startup, via `AsyncDatabricksSession._ensure_tables()`. You do **not** need to create them by hand. Make sure the schema exists so the first start succeeds.

In [ ]:
from databricks_ai_bridge.lakebase import LakebaseClient

with LakebaseClient(instance_name=lakebase_instance) as client:
    client.execute(f"CREATE SCHEMA IF NOT EXISTS {memory_schema};")
    print(f"✓ Schema '{memory_schema}' ready (the app creates the tables on first start).")

## 3. Action Plane schema

The Action Plane tables are created by the repo script `scripts/actions_schema_setup.py`, which also seeds the four L200 policies (all `requires_approval = true`). Run it from a terminal with your Lakebase env vars set:

```bash
export LAKEBASE_INSTANCE=<your-instance>
export LAKEBASE_SCHEMA=akzo_actions
uv run python scripts/actions_schema_setup.py
```

Or run the equivalent DDL from here. The cell below creates the schema only; the table DDL and policy seed live in the script so there is a single source of truth.

In [ ]:
with LakebaseClient(instance_name=lakebase_instance) as client:
    client.execute(f"CREATE SCHEMA IF NOT EXISTS {actions_schema};")
    print(f"✓ Schema '{actions_schema}' ready. Now run scripts/actions_schema_setup.py to create the tables and seed policies.")

## 4. Next

- Put the instance name in `.env` as `LAKEBASE_INSTANCE_NAME` (memory) and `LAKEBASE_INSTANCE` (Action Plane).
- Run `scripts/actions_schema_setup.py` to finish the Action Plane tables.
- After you deploy the app, grant its service principal access with `scripts/grant_lakebase_permissions.py` (see the **lakebase-setup** skill).
- Start the agent: `uv run start-app`.